# 🧠 Old Permic OCR — Adaptive YOLO Curriculum Training Engine

**Layer 3**: Adaptive YOLO training with checkpointing, remediation, and release pipeline.
Designed for Google Colab with **automatic resume** from disconnections.

## Pipeline Overview

```
Stage 1 → Train → Evaluate → [Remediate weak classes] → Accept → Release
Stage 2 → Train → Evaluate → [Remediate weak classes] → Accept → Release
  ...  (12 stages total, each building on the previous)
Stage 12 → Train → Evaluate → Accept → Final Release → Ready for real-data fine-tuning
```

## Key Features
- 🔄 **Colab-resilient**: every stage saved to disk; resume from any point after disconnection
- 🎯 **Per-class acceptance**: each character class must meet AP50 + Recall thresholds
- 🔧 **Adaptive remediation**: weak classes get targeted reserve-data fine-tuning (up to 3 rounds)
- 📦 **Auto-publish**: validated models packaged as `.ocrpkg` and pushed to results branch
- 🔍 **Regression testing**: previous-stage subsets evaluated to detect catastrophic forgetting

In [ ]:
# ── Cell 02: Runtime Inspection ───────────────────────────────────────────────
import subprocess, sys, platform, shutil, os

print('━' * 60)
print('  RUNTIME ENVIRONMENT')
print('━' * 60)
print(f'Python   : {sys.version.split()[0]}')
print(f'Platform : {platform.system()} {platform.machine()}')

# GPU info
try:
    import torch
    if torch.cuda.is_available():
        dev = torch.cuda.get_device_properties(0)
        print(f'GPU      : {dev.name} ({dev.total_memory / 1e9:.1f} GB VRAM)')
        print(f'CUDA     : {torch.version.cuda}')
        print(f'PyTorch  : {torch.__version__}')
    else:
        print('GPU      : ⚠️  Not available — training will be slow on CPU!')
except ImportError:
    try:
        gpu_info = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
            text=True, timeout=5
        ).strip()
        print(f'GPU      : {gpu_info}')
    except Exception:
        print('GPU      : ⚠️  nvidia-smi not found. Check runtime type.')

# RAM + Disk
try:
    import psutil
    ram = psutil.virtual_memory().total / 1e9
    print(f'RAM      : {ram:.1f} GB')
except ImportError:
    pass

disk = shutil.disk_usage('/content' if os.path.exists('/content') else '.')
print(f'Disk     : {disk.free / 1e9:.1f} GB free of {disk.total / 1e9:.1f} GB')
print('━' * 60)

In [ ]:
# ── Cell 03: GitHub Authentication ───────────────────────────────────────────
import os, getpass, stat, tempfile

def _get_github_token() -> str:
    """Obtain GitHub token securely. Never prints or stores it."""
    # 1. Colab Secrets (recommended)
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        if token:
            print('✅ Token loaded from Colab Secrets.')
            return token
    except Exception:
        pass
    # 2. Environment variable
    token = os.environ.get('GITHUB_TOKEN', '')
    if token:
        print('✅ Token loaded from environment variable.')
        return token
    # 3. Secure prompt
    print('Enter your GitHub Personal Access Token (repo + contents scope required).')
    return getpass.getpass('Token (hidden): ')

_TOKEN = _get_github_token()
os.environ['GITHUB_TOKEN'] = _TOKEN

# GIT_ASKPASS helper — script echoes token without exposing it in process args
_askpass = tempfile.NamedTemporaryFile(
    mode='w', suffix='.sh', delete=False, prefix='/tmp/git_askpass_'
)
_askpass.write('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
_askpass.close()
os.chmod(_askpass.name, stat.S_IRWXU)
os.environ['GIT_ASKPASS'] = _askpass.name

print('✅ Authentication configured. Token is in memory only — never printed.')

In [ ]:
# ── Cell 04: Repository & Path Configuration ─────────────────────────────────
import os

TRAINING_REPO     = "https://github.com/Emran025/ocroldpermic"
CODE_BRANCH      = "main"                 # Package code, fonts, and notebooks
IMAGE_BRANCH      = "colab-generated-images"  # Generated image datasets only
CHECKPOINT_BRANCH = "colab-checkpoints"    # Training checkpoints and state only
RESULT_BRANCH    = "colab-results"

DATASET_ROOT  = "/content/datasets"
REPO_DIR      = "/content/repo"           # Code + font/svg checkout
IMAGE_REPO_DIR = "/content/image-repo"          # Image branch checkout
CHECKPOINT_REPO_DIR = "/content/checkpoint-repo"  # Checkpoint branch checkout
RESULT_REPO_DIR = "/content/result-repo"  # Published result branch checkout
GLYPH_ROOT    = os.path.join(REPO_DIR, "font", "svg")
RUN_ROOT      = "/content/runs"
SESSION_FILE  = "/content/training_session.json"
METRICS_ROOT  = "/content/metrics"
RELEASE_ROOT  = "/content/releases"
BASE_MODEL    = "yolo11n.pt"
GENERATION_MODE = "full"

# Decoupled 2-Machine & Auto-Run Settings
AUTO_APPROVE       = True   # Non-blocking approval in training
WAIT_FOR_STAGES    = True   # Wait for stage from generator if not yet available
POLL_INTERVAL_SEC  = 30     # Interval to poll Git for newly pushed stages

for d in [RUN_ROOT, METRICS_ROOT, RELEASE_ROOT]:
    os.makedirs(d, exist_ok=True)

print("Configuration loaded:")
print(f"  Repository   : {TRAINING_REPO}")
print(f"  Branches     : code={CODE_BRANCH}  images={IMAGE_BRANCH}  checkpoint={CHECKPOINT_BRANCH}  results={RESULT_BRANCH}")
print(f"  Code/fonts   : {REPO_DIR}")
print(f"  Checkpoints  : {CHECKPOINT_REPO_DIR}")
print(f"  Dataset      : {DATASET_ROOT}")
print(f"  Base model   : {BASE_MODEL}")
print(f"  Auto approve : {AUTO_APPROVE}")
print(f"  Wait stages  : {WAIT_FOR_STAGES} (poll every {POLL_INTERVAL_SEC}s)")


In [ ]:
# ── Cell 05: Install Dependencies ────────────────────────────────────────────
import subprocess, sys

def _install(pkg, label=None):
    label = label or pkg.split('/')[-1]
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    icon = '✅' if r.returncode == 0 else '❌'
    print(f'{icon} {label}')
    if r.returncode != 0:
        print(r.stderr[-500:])

_install('git+https://github.com/Emran025/ocroldpermic@main',
         'ocroldpermic')
_install('ultralytics', 'ultralytics')
_install('pyyaml',  'pyyaml')
_install('psutil',  'psutil')

print('\nAll dependencies installed.')

In [ ]:
# ── Cell 06: Import Training Engine ──────────────────────────────────────────
from historical_glyph_training import (
    TrainingConfig, TrainingSession, StageStatus
)
from historical_glyph_training.config.training_config import (
    AcceptanceCriteria, RemediationConfig, CheckpointPolicy
)
from historical_glyph_training.dataset import DatasetSplitter, DatasetLoader, ReservePool
from historical_glyph_training.training import (
    YoloTrainer, Evaluator, RemediationEngine, PlateauDetector, ResourceMonitor
)
from historical_glyph_training.training.regression import RegressionEvaluator
from historical_glyph_training.checkpoint import CheckpointManager
from historical_glyph_training.checkpoint.metadata import CheckpointMetadata
from historical_glyph_training.release.release import ReleaseManager
from historical_glyph_training.git.git_ops import GitManager
from historical_glyph_training.reporting import TrainingReporter
from historical_glyph_training.audit import AuditTrail
from historical_glyph_training.state.stage_state import StageStatus, CheckpointRef

print('✅ Training engine imported successfully.')

In [ ]:
# ── Cell 07: Resource Detection ───────────────────────────────────────────────
monitor = ResourceMonitor()
profile = monitor.detect(dataset_root=DATASET_ROOT)
monitor.log_profile(profile)

In [ ]:
# ── Cell 08: Training Configuration ──────────────────────────────────────────

# Adjust epochs by mode for faster iteration
_EPOCHS_BY_MODE = {'dev': 10, 'medium': 30, 'full': 50}

config = TrainingConfig(
    model_repository=TRAINING_REPO,
    model_reference=CHECKPOINT_BRANCH,
    model_name=BASE_MODEL.replace('.pt', ''),
    epochs_per_stage=_EPOCHS_BY_MODE.get(GENERATION_MODE, 50),
    batch_size=profile.suggested_batch_size,
    image_size=640,
    workers=profile.suggested_workers,
    device=profile.device_string,
    seed=42,
    amp=True,
    dataset_root=DATASET_ROOT,
    num_stages=12,
    train_ratio=0.75,
    val_ratio=0.15,
    reserve_ratio=0.10,
    checkpoint_policy=CheckpointPolicy.BEST_ONLY,
    acceptance=AcceptanceCriteria(
        global_map50=0.90,
        global_map50_95=0.70,
        global_recall=0.85,
        per_class_ap50=0.70,
        per_class_recall=0.65,
        regression_tolerance=0.05,
        strict=True,
    ),
    remediation=RemediationConfig(
        max_rounds=3,
        max_extra_epochs=20,
        min_improvement_delta=0.01,
        patience=5,
        reserve_ratio=0.10,
        min_reserve_per_class=20,
        diversity_selection=True,
    ),
    checkpoint_branch=CHECKPOINT_BRANCH,
    release_branch=RESULT_BRANCH,
    run_root=RUN_ROOT,
    session_file=SESSION_FILE,
    metrics_root=METRICS_ROOT,
    release_root=RELEASE_ROOT,
    generation_mode=GENERATION_MODE,
    verbose=True,
)

print('Training configuration:')
print(f'  Epochs/stage  : {config.epochs_per_stage}')
print(f'  Batch size    : {config.batch_size}')
print(f'  Image size    : {config.image_size}')
print(f'  Device        : {config.device}')
print(f'  Acceptance    : mAP50>={config.acceptance.global_map50:.2f}  per-class AP50>={config.acceptance.per_class_ap50:.2f}')
print(f'  Remediation   : max {config.remediation.max_rounds} rounds × {config.remediation.max_extra_epochs} epochs')

In [ ]:
# ── Cell 09: Dataset Discovery (Decoupled Stage Detection) ────────────────────
# The trainer reads only the image branch for datasets. Training state and
# checkpoints are written separately by GitManager to colab-checkpoints.
def _sync_branch(url, branch, target):
    import shutil, subprocess
    target = Path(target)
    if (target / ".git").is_dir():
        subprocess.run(["git", "-C", str(target), "fetch", "origin", branch], check=True)
        subprocess.run(["git", "-C", str(target), "checkout", "-B", branch, f"origin/{branch}"], check=True)
    else:
        if target.exists(): shutil.rmtree(target)
        subprocess.run(["git", "clone", "--branch", branch, "--depth", "1", url, str(target)], check=True)

_sync_branch(TRAINING_REPO, IMAGE_BRANCH, IMAGE_REPO_DIR)
loader = DatasetLoader(dataset_root=str(Path(IMAGE_REPO_DIR) / "datasets"))

print('Discovering available stage datasets from the independent image branch...')
print('─' * 60)
stage_ok = {}
for stage_id in range(1, 13):
    try:
        stage_dir = loader.discover_stage(stage_id)
        count, errors = loader.verify_integrity(stage_dir)
        icon = '✅' if not errors else '⚠️ '
        stage_ok[stage_id] = count > 0
        print(f'  {icon} Stage {stage_id:02d}: {count:5d} image/label pairs | {stage_dir}')
    except FileNotFoundError:
        stage_ok[stage_id] = False

ready = sum(v for v in stage_ok.values())
print('─' * 60)
print(f'Stages currently ready: {ready} / 12')
if ready == 0:
    print('ℹ️  No stages downloaded yet. The image branch will be synchronized dynamically when training starts.')
elif ready < 12:
    print(f'ℹ️  {ready} stage(s) ready for training immediately. Future stages will be polled from git.')
else:
    print('✅ All 12 stages ready!')


In [ ]:
# ── Cell 10: Dataset Splits (train / val / reserve) ───────────────────────────
import json
from pathlib import Path

splitter = DatasetSplitter(
    train_ratio=config.train_ratio,
    val_ratio=config.val_ratio,
    reserve_ratio=config.reserve_ratio,
    seed=config.seed,
    min_reserve_per_class=config.remediation.min_reserve_per_class,
)

split_manifests = {}
print('Loading existing dataset splits...')
for stage_id in range(1, 13):
    try:
        stage_dir = loader.discover_stage(stage_id)
        manifest_path = Path(stage_dir) / 'split_manifest.json'
        if manifest_path.exists():
            split_manifests[stage_id] = splitter.load_manifest(str(manifest_path))
            m = split_manifests[stage_id]
            print(f'  Stage {stage_id:02d}: loaded split train={len(m.train_files)} val={len(m.val_files)} reserve={len(m.reserve_files)}')
        else:
            m = splitter.split(stage_dir, stage_id)
            splitter.save_manifest(m, str(manifest_path))
            split_manifests[stage_id] = m
            print(f'  Stage {stage_id:02d}: created split train={len(m.train_files)} val={len(m.val_files)} reserve={len(m.reserve_files)}')
    except FileNotFoundError:
        pass

print(f'\nSplits currently prepared: {len(split_manifests)} stages (additional stages split automatically on demand)')


In [ ]:
# ── Cell 11: Resume State Detection ──────────────────────────────────────────
session = TrainingSession.load_or_create(SESSION_FILE)
print(session.summary())
print()
print(f'→  Next stage to train : {session.current_stage_id}')
print(f'   Last completed stage: {session.last_completed_stage}')

# Determine starting weights. A fresh run bootstraps from the latest published
# result (useful prior training), not from the base model. Later resumed stages
# continue from their accepted local checkpoint.
def _latest_published_weights(result_root):
    result_root = Path(result_root)
    pointer = result_root / 'artifacts' / 'published' / 'latest.json'
    if not pointer.is_file():
        raise FileNotFoundError(
            'No published result found on colab-results; cannot bootstrap training.'
        )
    data = json.loads(pointer.read_text(encoding='utf-8'))
    release_path = data.get('release_path')
    if not release_path:
        raise RuntimeError('colab-results latest.json has no release_path')
    release_file = result_root / release_path
    release = json.loads(release_file.read_text(encoding='utf-8'))
    weight_path = release.get('web_weight', {}).get('path')
    if not weight_path:
        raise RuntimeError('Published result has no web_weight path')
    weights = release_file.parent / weight_path
    if not weights.is_file():
        raise FileNotFoundError(f'Published bootstrap weights missing: {weights}')
    return str(weights), release.get('release_id', '')

_sync_branch(TRAINING_REPO, RESULT_BRANCH, RESULT_REPO_DIR)
if session.last_completed_stage:
    last_record = session.get_stage(session.last_completed_stage)
    best_ckpt = last_record.best_checkpoint()
    if best_ckpt and __import__('os').path.exists(best_ckpt.path):
        starting_weights = best_ckpt.path
        starting_provenance = f'local checkpoint stage {session.last_completed_stage:02d}'
    else:
        starting_weights, starting_provenance = _latest_published_weights(RESULT_REPO_DIR)
else:
    starting_weights, starting_provenance = _latest_published_weights(RESULT_REPO_DIR)
print(f'\n📦 Starting weights: {starting_weights}')
print(f'   Provenance: {starting_provenance}')

In [ ]:
# ── Cell 12: Git Manager Setup ────────────────────────────────────────────────
import os

git_manager = GitManager(
    repo_url=TRAINING_REPO,
    checkpoint_branch=CHECKPOINT_BRANCH,
    release_branch=RESULT_BRANCH,
    work_dir=CHECKPOINT_REPO_DIR,
)
git_manager.setup(token=os.environ.get('GITHUB_TOKEN', ''))

# Shared helpers
audit = AuditTrail(f'{METRICS_ROOT}/audit.jsonl')
release_manager = ReleaseManager(release_root=RELEASE_ROOT, config=config)
regressor = RegressionEvaluator(regression_root=f'{METRICS_ROOT}/regression',
                                subset_size=config.regression_subset_size)

print('✅ Git manager ready.')
print('✅ Audit trail ready.')
print('✅ Release manager ready.')

In [ ]:
# ── Cell 13: Discover generated stages from checkpoint branch ────────────────
from pathlib import Path
import re, time

def available_generated_stages():
    found = set()
    roots = [Path(DATASET_ROOT), Path(IMAGE_REPO_DIR) / 'datasets', Path(IMAGE_REPO_DIR)]
    for root in roots:
        if not root.exists():
            continue
        for candidate in root.glob('stage_*'):
            match = re.fullmatch(r'stage_?(\d+)', candidate.name)
            if match and candidate.is_dir():
                try:
                    count, _ = loader.verify_integrity(str(candidate))
                    if count > 0:
                        found.add(int(match.group(1)))
                except Exception:
                    pass
        for candidate in root.glob('stage_*/*'):
            match = re.fullmatch(r'stage_?(\d+)', candidate.name)
            if match and candidate.is_dir():
                try:
                    count, _ = loader.verify_integrity(str(candidate))
                    if count > 0:
                        found.add(int(match.group(1)))
                except Exception:
                    pass
    return sorted(found)

def latest_generated_stage(wait=True, max_wait_sec=7200):
    started = time.time()
    while True:
        try:
            _sync_branch(TRAINING_REPO, IMAGE_BRANCH, IMAGE_REPO_DIR)
        except Exception as exc:
            print(f'[Sync] checkpoint branch notice: {exc}')
        stages = available_generated_stages()
        if stages:
            print(f'Image branch stages: {stages}; latest={stages[-1]}')
            return stages[-1]
        if not wait or time.time() - started > max_wait_sec:
            return None
        print('Waiting for generated stages on colab-generated-images...')
        time.sleep(POLL_INTERVAL_SEC)

print('Dynamic checkpoint-stage discovery ready.')


In [ ]:
# ── Cell 13: Approval Gate Helper ────────────────────────────────────────────
import time

def approval_gate(title: str, metrics_summary: str = '', timeout: int = 30) -> bool:
    """
    Non-blocking approval gate with Jupyter event-loop integration and AUTO_APPROVE support.
    Returns True=accept, False=reject, 'force'=force-accept.
    """
    print(f"\n{'═'*60}")
    print(f'  {title}')
    print(f"{'═'*60}")
    if metrics_summary:
        print(metrics_summary)
    print()

    auto = globals().get('AUTO_APPROVE', False)
    if auto:
        print('  ⚡ Auto-accepted (AUTO_APPROVE = True)')
        return True

    try:
        from IPython import get_ipython
        import ipywidgets as w
        from IPython.display import display

        ip = get_ipython()
        decision = [None]
        btn_ok    = w.Button(description='✅ Accept Stage', button_style='success', layout=w.Layout(width='160px'))
        btn_force = w.Button(description='⚡ Force Accept', button_style='warning', layout=w.Layout(width='160px'))
        btn_stop  = w.Button(description='🛑 Reject & Stop', button_style='danger', layout=w.Layout(width='160px'))
        out = w.Output()

        def _accept(_): decision[0] = True;   out.clear_output(); print('Accepted ✅')
        def _force(_):  decision[0] = 'force';out.clear_output(); print('Force accepted ⚡')
        def _reject(_): decision[0] = False;  out.clear_output(); print('Rejected 🛑')

        btn_ok.on_click(_accept)
        btn_force.on_click(_force)
        btn_stop.on_click(_reject)
        display(w.HBox([btn_ok, btn_force, btn_stop]), out)

        start_time = time.time()
        while decision[0] is None:
            if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, 'do_one_iteration'):
                ip.kernel.do_one_iteration()
            time.sleep(0.05)
            if timeout and (time.time() - start_time > timeout):
                print(f'\n  ⏱ Timeout reached ({timeout}s) — auto-accepting stage to continue.')
                return True

        return decision[0]
    except Exception:
        try:
            resp = input('Accept stage? [y/force/n] (default y): ').strip().lower()
            if resp in ('y', 'yes', ''): return True
            if resp == 'force':          return 'force'
            return False
        except Exception:
            return True

print('✅ Approval gate helper ready.')


In [ ]:
# ── Cell 14: Stage Execution Engine & Decoupled Git Poller ─────────────────────
import os, time
from pathlib import Path

# Build class_names list from curriculum stage definitions
try:
    from historical_glyph_studio import GlyphStudio
    _studio = GlyphStudio(glyph_root=GLYPH_ROOT)
    _repo = _studio.repository
    CLASS_NAMES = _studio.available_class_names()
except Exception:
    CLASS_NAMES = [f'U+{0x10350 + i:04X}' for i in range(38)]

def ensure_stage_ready(
    stage_id: int,
    loader,
    splitter,
    git_manager,
    wait: bool = True,
    poll_interval: int = 30,
    max_wait_sec: int = 7200,
):
    """
    Ensure a stage is available locally for training:
    1. Check local dataset directories.
    2. Pull latest commits from colab-generated-images (production data) branch.
    3. If wait=True and still not found, poll git until the generator pushes it.
    4. Load or compute the split_manifest.
    """
    from historical_glyph_curriculum.validation.dataset import DatasetValidator

    def validate_stage_contract(stage_dir):
        stage_dir = Path(stage_dir)
        manifest_path = stage_dir / 'manifest.json'
        if manifest_path.exists():
            data = json.loads(manifest_path.read_text(encoding='utf-8'))
            contract = data.get('dataset_contract', {})
            if contract:
                if contract.get('image_format') != 'png' or contract.get('label_format') != 'yolo-5-normalized':
                    raise RuntimeError(f'Stage {stage_id:02d} has an incompatible image/label format: {contract}')
                if int(contract.get('class_count', len(CLASS_NAMES))) != len(CLASS_NAMES):
                    raise RuntimeError(
                        f'Stage {stage_id:02d} class count {contract.get("class_count")} does not match model alphabet {len(CLASS_NAMES)}'
                    )
        report = DatasetValidator().validate(
            stage_dir / 'images', stage_dir / 'labels', expected_class_count=len(CLASS_NAMES)
        )
        if not report.is_valid or report.missing_labels or report.missing_images:
            raise RuntimeError(f'Stage {stage_id:02d} failed image/label compatibility validation: {report.issues[:3]}')
        non_png = [p for p in (stage_dir / 'images').iterdir() if p.is_file() and p.suffix.lower() != '.png']
        if non_png:
            raise RuntimeError(f'Stage {stage_id:02d} contains non-PNG images: {non_png[:3]}')

    t0 = time.time()
    while True:
        try:
            stage_dir = loader.discover_stage(stage_id)
            count, _ = loader.verify_integrity(stage_dir)
            if count > 0:
                validate_stage_contract(stage_dir)
                print(f'  ✅ Stage {stage_id:02d} dataset detected and compatible ({count} pairs) at {stage_dir}')
                break
        except FileNotFoundError:
            pass

        print(f'  [Sync] Pulling latest image-branch data for Stage {stage_id:02d}...')
        try:
            _sync_branch(TRAINING_REPO, IMAGE_BRANCH, IMAGE_REPO_DIR)
        except Exception as e:
            print(f'  [Sync] Image branch notice: {e}')

        try:
            stage_dir = loader.discover_stage(stage_id)
            count, _ = loader.verify_integrity(stage_dir)
            if count > 0:
                validate_stage_contract(stage_dir)
                print(f'  ✅ Stage {stage_id:02d} dataset found and compatible after sync ({count} pairs)')
                break
        except FileNotFoundError:
            pass

        if not wait:
            return None

        elapsed = int(time.time() - t0)
        if elapsed > max_wait_sec:
            print(f'  ⏱ Max wait time ({max_wait_sec}s) exceeded for Stage {stage_id:02d}.')
            return None

        print(f'  ⏳ Waiting for Stage {stage_id:02d} to be generated & pushed... ({elapsed}s elapsed, next check in {poll_interval}s)')
        time.sleep(poll_interval)

    stage_dir_path = Path(stage_dir)
    stage_manifest_file = stage_dir_path / 'split_manifest.json'
    if stage_manifest_file.exists():
        return splitter.load_manifest(str(stage_manifest_file))

    m = splitter.split(str(stage_dir_path), stage_id)
    splitter.save_manifest(m, str(stage_manifest_file))
    return m

def run_stage(
    stage_id: int,
    current_weights: str,
    session: TrainingSession,
    config: TrainingConfig,
    split_manifests: dict,
    loader: DatasetLoader,
    git_manager: GitManager,
    wait_for_generation: bool = True,
) -> tuple[str, bool]:
    """Execute the complete training lifecycle for one stage."""
    record = session.get_stage(stage_id)
    if record.is_terminal and record.status in (StageStatus.ACCEPTED, StageStatus.RELEASED):
        print(f'\n✅ Stage {stage_id:02d} already {record.status.value}. Skipping.')
        best_ckpt = record.best_checkpoint()
        return (best_ckpt.path if best_ckpt and os.path.exists(best_ckpt.path) else current_weights), True

    if stage_id not in split_manifests:
        manifest = ensure_stage_ready(stage_id, loader, splitter, git_manager, wait=wait_for_generation)
        if manifest is None:
            print(f'\n❌ Stage {stage_id:02d}: dataset not available yet.')
            return current_weights, False
        split_manifests[stage_id] = manifest

    print(f"\n{'━'*60}")
    print(f'  📚 STAGE {stage_id:02d} — Training')
    print(f"{'━'*60}")

    # ── 1. Prepare dataset ──────────────────────────────────────────────────
    session.start_stage(stage_id)
    manifest = split_manifests[stage_id]
    work_dir = f'{RUN_ROOT}/stage_{stage_id:02d}/splits'

    stage_dataset = loader.load_stage(stage_id, manifest, CLASS_NAMES, work_dir=work_dir)
    record.train_samples = stage_dataset.train_images
    record.val_samples = stage_dataset.val_images

    # ── 2. Save regression subset ───────────────────────────────────────────
    regressor.save_regression_subset(stage_id, manifest.val_files, CLASS_NAMES, seed=config.seed)

    # ── 3. Set up checkpoint manager ────────────────────────────────────────
    run_dir = f'{RUN_ROOT}/stage_{stage_id:02d}/train'
    ckpt_mgr = CheckpointManager(run_dir, config.checkpoint_policy, config.checkpoint_every_n)
    plateau   = PlateauDetector(patience=5, min_delta=0.005)
    audit.log('stage_start', stage_id, weights=current_weights)

    # ── 4. Epoch callback ───────────────────────────────────────────────────
    def on_epoch(epoch, metrics):
        is_best = metrics.map50 > record.best_map50
        plateau.update(epoch, metrics.map50)

        if is_best:
            record.best_epoch  = epoch
            record.best_map50  = metrics.map50
            record.best_map50_95 = metrics.map50_95
        record.epochs_completed = epoch

        print(f'  Epoch {epoch:3d}/{config.epochs_per_stage} | {metrics.summary()}')
        audit.log_epoch(stage_id, epoch, metrics.map50, metrics.map50_95, is_best)

        if plateau.is_plateau():
            print(f'  ⏸  Plateau detected at epoch {epoch} (no improvement for {plateau.patience} epochs).')
            return False  # Stop training
        return True

    # ── 5. Train ────────────────────────────────────────────────────────────
    epoch_publish_dir = Path(RUN_ROOT) / 'published_checkpoints'
    epoch_publish_dir.mkdir(parents=True, exist_ok=True)

    def on_epoch_checkpoint(epoch, weights_path, metrics):
        """Persist every epoch locally and push it only to colab-checkpoints."""
        is_best = epoch == record.best_epoch
        metadata = CheckpointMetadata(
            stage=stage_id, epoch=epoch, global_step=epoch,
            map50=metrics.map50, map50_95=metrics.map50_95,
            precision=metrics.precision, recall=metrics.recall,
            train_loss=metrics.train_loss, val_loss=0.0,
            class_metrics=record.class_ap50,
            training_config={'epochs_per_stage': config.epochs_per_stage, 'image_size': config.image_size},
            dataset_stage=stage_id, dataset_version=manifest.created_at,
            model_source_commit=git_manager.current_commit(),
            random_seed=config.seed, is_best=is_best,
        )
        local_path = ckpt_mgr.save(weights_path, metadata)
        published_weights = epoch_publish_dir / f'stage_{stage_id:02d}_epoch_{epoch:03d}.pt'
        published_meta = epoch_publish_dir / f'stage_{stage_id:02d}_epoch_{epoch:03d}.json'
        import shutil
        shutil.copy2(local_path, published_weights)
        published_meta.write_text(json.dumps(metadata.to_dict(), indent=2), encoding='utf-8')
        git_manager.push_checkpoint(
            stage_id=stage_id, epoch=epoch,
            files=[str(published_weights), str(published_meta)], is_best=is_best,
        )
        print(f'  Epoch {epoch:03d} checkpoint pushed to {CHECKPOINT_BRANCH}')

    record.transition(record.status.__class__.TRAINING)
    session.save()

    trainer = YoloTrainer(
        model_weights=current_weights,
        stage_dataset=stage_dataset,
        config=config,
        run_dir=run_dir,
    )
    try:
        best_weights = trainer.train(on_epoch_end=on_epoch, on_epoch_checkpoint=on_epoch_checkpoint)
    except Exception as e:
        print(f'  ❌ Training failed: {e}')
        session.mark_failed(stage_id, str(e))
        return current_weights, False

    # ── 6. Evaluate ─────────────────────────────────────────────────────────
    record.transition(record.status.__class__.EVALUATING)
    session.save()

    evaluator = Evaluator(best_weights, stage_dataset.data_yaml)
    eval_result = evaluator.evaluate(config)
    print(f'\n  Evaluation result:')
    print(f'  {eval_result.summary()}')

    record.final_map50     = eval_result.map50
    record.final_map50_95  = eval_result.map50_95
    record.final_recall    = eval_result.recall
    record.final_precision = eval_result.precision
    record.class_ap50      = {c.class_name: c.ap50 for c in eval_result.class_metrics}

    # ── 7. Remediation ──────────────────────────────────────────────────────
    reserve_pool  = ReservePool(manifest.reserve_files, stage_id)
    rem_engine    = RemediationEngine(reserve_pool, config, run_dir)

    for round_num in range(1, config.remediation.max_rounds + 1):
        weak = config.acceptance.weak_classes(eval_result.class_metrics)
        passes_global = config.acceptance.is_globally_passing(eval_result)

        if not weak and passes_global:
            break  # All criteria met

        if not weak:
            break  # Only global metric issue; can't remediate further

        weak_names = [c.class_name for c in weak]
        print(f'\n  🔧 Remediation round {round_num}/{config.remediation.max_rounds}')
        print(f'     Weak classes: {weak_names[:5]}{"..." if len(weak_names) > 5 else ""}')

        record.transition(record.status.__class__.REMEDIATING)
        audit.log_remediation_start(stage_id, round_num, weak_names)

        remediation_epoch_offset = record.epochs_completed
        best_weights, rem_result = rem_engine.run_remediation_round(
            best_weights, stage_dataset, weak, round_num, eval_result,
            on_epoch_checkpoint=lambda epoch, path, metrics: on_epoch_checkpoint(
                remediation_epoch_offset + epoch, path, metrics
            )
        )
        evaluator = Evaluator(best_weights, stage_dataset.data_yaml)
        eval_result = evaluator.evaluate(config)

        audit.log_remediation_end(stage_id, round_num, rem_result.map50_before, rem_result.map50_after)

        if rem_result.catastrophic_forgetting:
            print(f'  ⚠️  Catastrophic forgetting detected! Reverting remediation.')
            break

        if not rem_result.improved:
            print(f'  ℹ️  No improvement in round {round_num}. Stopping remediation.')
            break

    # ── 8. Approval gate ────────────────────────────────────────────────────
    passes = config.acceptance.is_globally_passing(eval_result)
    remaining_weak = config.acceptance.weak_classes(eval_result.class_metrics)
    summary = (
        f'  mAP50        : {eval_result.map50:.4f}  (need ≥ {config.acceptance.global_map50:.2f})\n'
        f'  mAP50-95     : {eval_result.map50_95:.4f}\n'
        f'  Recall       : {eval_result.recall:.4f}\n'
        f'  Global pass  : {"✅ YES" if passes else "❌ NO"}\n'
        f'  Weak classes : {len(remaining_weak)} remaining'
    )
    decision = approval_gate(f'Stage {stage_id:02d} — Accept?', summary)

    if decision is False:
        session.reject_stage(stage_id, 'User rejected')
        audit.log_stage_rejected(stage_id, 'user_rejected')
        return current_weights, False

    # ── 9. Release ──────────────────────────────────────────────────────────
    print(f'\n  📦 Releasing stage {stage_id:02d}...')
    try:
        release_record = release_manager.promote(
            stage_id=stage_id,
            model_weights=best_weights,
            class_names=CLASS_NAMES,
            eval_result=eval_result,
            dataset_version=manifest.created_at,
            model_source_commit=git_manager.current_commit(),
            git_manager=git_manager,
        )
        session.mark_released(stage_id, release_record.commit_hash, release_record.branch)
        audit.log_release(stage_id, release_record.version,
                          release_record.commit_hash, release_record.branch)
        print(f'  ✅ Stage {stage_id:02d} released! Package: {release_record.version}')
    except Exception as e:
        print(f'  ⚠️  Release failed ({e}). Marking as ACCEPTED only.')
        session.accept_stage(stage_id, f'Release failed: {e}')

    # Cleanup temporary split dirs to save disk space
    loader.cleanup_training_workspace(stage_dataset)

    return best_weights, True

print('✅ run_stage() and ensure_stage_ready() functions ready.')


In [ ]:
# ── Cell 18: Regression Testing ───────────────────────────────────────────────
print('Running regression tests across all completed stages...')
print('(Checks current model still recognises characters from earlier stages)')
print()

try:
    _last_stage = session.last_completed_stage or 1
    regression_results = regressor.evaluate_regression(
        starting_weights, _last_stage, config
    )

    if regression_results:
        print('─' * 50)
        for _sid, _map50 in sorted(regression_results.items()):
            _icon = '✅' if _map50 >= 0.80 else ('⚠️ ' if _map50 >= 0.0 else '❌')
            print(f'  {_icon} Stage {_sid:02d} subset: mAP50={_map50:.3f}')
        print('─' * 50)

        _regressed = regressor.check_regression({}, regression_results)
        if _regressed:
            print(f'⚠️  Regression detected in stages: {_regressed}')
        else:
            print('✅ No regression detected!')
    else:
        print('ℹ️  No previous-stage subsets to evaluate yet.')
        regression_results = {}
except Exception as e:
    print(f'⚠️  Regression test failed: {e}')
    regression_results = {}

In [ ]:
# ── Cell 19: Final Report ─────────────────────────────────────────────────────
reporter = TrainingReporter(session=session, metrics_root=METRICS_ROOT)
report = reporter.generate_final_report(
    regression_results=regression_results,
    current_weights=starting_weights,
)
print(report)

# Readiness check
_all_done = all(
    session.get_stage(sid).status.is_success
    for sid in range(1, 13)
    if sid in split_manifests
)

print()
if _all_done:
    print('═' * 60)
    print('  🎉 CURRICULUM TRAINING COMPLETE!')
    print('  ✅ All 12 stages accepted and released.')
    print('  📱 Models available as .ocrpkg in the Flutter app.')
    print('  🔬 Next: fine-tune on real Old Permic manuscript images.')
    print('═' * 60)
else:
    _pending = [sid for sid in range(1, 13) if not session.get_stage(sid).status.is_success]
    print(f'⚠️  Pending stages: {_pending}')
    print('Re-run the corresponding stage cells to complete training.')

# Save audit trail summary
all_events = audit.read_all()
print(f'\nAudit trail: {len(all_events)} events logged to {METRICS_ROOT}/audit.jsonl')

In [ ]:
# ── Dynamic curriculum execution ─────────────────────────────────────────────
latest = latest_generated_stage(wait=WAIT_FOR_STAGES)
if latest is None:
    raise RuntimeError('No generated stage is available on the image branch.')
_current_w = starting_weights
# Do not retrain completed stages: resume from the first model stage that is
# not terminal, while the image branch may already contain later stages.
_sid = max(1, session.current_stage_id or ((session.last_completed_stage or 0) + 1))
if _sid > latest:
    print(f'✅ Model already covers image stages through {latest:02d}. Nothing new to train.')
while _sid <= latest:
    _current_w, _ok = run_stage(
        _sid, _current_w, session, config, split_manifests, loader, git_manager,
        wait_for_generation=WAIT_FOR_STAGES
    )
    if not _ok:
        print(f'Stage {_sid:02d} stopped. Check data or metrics before continuing.')
        break
    starting_weights = _current_w
    refreshed = latest_generated_stage(wait=False)
    if refreshed is not None:
        latest = max(latest, refreshed)
    _sid += 1
print(f'Training run finished through latest discovered stage: {latest:02d}')
